## KOBIS API KEY로 영화 상세정보 수집

### 테스트 : 영화 1개 출력 

In [1]:
import requests

# 1. 기본 설정
API_KEY = "e60a0576c6cdfd42b0776f86f25ac236" # 팀원분 코드에 있던 테스트 키
TEST_MOVIE_CD = "20124079" # 테스트용 영화코드 (명량)

url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"

params = {
    "key": API_KEY,
    "movieCd": TEST_MOVIE_CD
}

# 2. API 호출
print(f"[{TEST_MOVIE_CD}] 영화 상세정보 요청 중...")
res = requests.get(url, params=params)
data = res.json()

# 3. 데이터가 정상적으로 왔는지 확인
if "movieInfoResult" in data:
    movie_info = data["movieInfoResult"]["movieInfo"]
    
    print("\n데이터 추출 테스트:")
    print("-" * 40)
    print("영화명(국문):", movie_info.get("movieNm"))
    print("개봉일:", movie_info.get("openDt"))
    print("상영시간:", movie_info.get("showTm"), "분")

    genres = [g["genreNm"] for g in movie_info.get("genres", [])]
    print("장르:", genres)
    
    directors = [d["peopleNm"] for d in movie_info.get("directors", [])]
    print("감독:", directors) 
    
    actors = [a["peopleNm"] for a in movie_info.get("actors", [])][:5]
    print("배우:", actors)

    companys = [c["companyNm"] for c in movie_info.get("companys", [])]
    print("참여 영화사:", companys)
    print("-" * 60)

else:
    print("데이터를 불러오지 못했습니다. 에러 메시지:", data)

[20124079] 영화 상세정보 요청 중...

데이터 추출 테스트:
----------------------------------------
영화명(국문): 광해, 왕이 된 남자
개봉일: 20120913
상영시간: 131 분
장르: ['사극', '드라마']
감독: ['추창민']
배우: ['이병헌', '류승룡', '한효주', '장광', '김인권']
참여 영화사: ['리얼라이즈픽쳐스(주)', '(주)씨제이이엔엠', '(주)씨제이이엔엠', '(주)씨제이이엔엠', '비엠씨영화전문투자조합', '컴퍼니케이파트너스 콘텐츠 전문투자조합', '그린손해보험(주)', '이수창업투자(주)', '소빅창업투자(주)', '에스크베리타스자산운용(주)', 'CJ창업투자(주)', 'MVP창투문화산업투자조합', '(유)동문파트너즈', '씨제이엔터테인먼트', '(사)한국농아인협회', '한국시각장애인연합회']
------------------------------------------------------------


### 테스트 : 영화 3개 출력

In [3]:
import requests
import pandas as pd

# 1. 기본 설정
API_KEY = "e60a0576c6cdfd42b0776f86f25ac236" 
url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"

# 테스트용 영화
sample_movies = ["20124079", "20183782", "20182530"] 

temp_results = []
print("테스트 시작!\n")


for movie_cd in sample_movies:
    params = {"key": API_KEY, "movieCd": movie_cd}
    res = requests.get(url, params=params)
    data = res.json()
    
    if "movieInfoResult" in data:
        info = data["movieInfoResult"]["movieInfo"]
        
        movie_dict = {
            "movieCd": movie_cd,
            "movieNm": info.get("movieNm", ""),
            "openDt": info.get("openDt", ""),
            "genres": [g["genreNm"] for g in info.get("genres", [])],
            "directors": [d["peopleNm"] for d in info.get("directors", [])],
            "actors": [a["peopleNm"] for a in info.get("actors", [])][:5] # 배우는 앞 5명만!
        }
        temp_results.append(movie_dict)
        print(f"'{info.get('movieNm')}' 수집 완료!")

# Pandas 데이터프레임으로 변환
df_sample = pd.DataFrame(temp_results)

print("\n최종 데이터프레임 확인:")
print(df_sample)

테스트 시작!

'광해, 왕이 된 남자' 수집 완료!
'기생충' 수집 완료!
'극한직업' 수집 완료!

최종 데이터프레임 확인:
    movieCd      movieNm    openDt     genres directors  \
0  20124079  광해, 왕이 된 남자  20120913  [사극, 드라마]     [추창민]   
1  20183782          기생충  20190530      [드라마]     [봉준호]   
2  20182530         극한직업  20190123      [코미디]     [이병헌]   

                      actors  
0   [이병헌, 류승룡, 한효주, 장광, 김인권]  
1  [송강호, 이선균, 조여정, 최우식, 박소담]  
2   [류승룡, 이하늬, 진선규, 이동휘, 공명]  


배우는 일단 5명으로 출력 제한

In [4]:
display(df_sample)

,movieCd,movieNm,openDt,genres,directors,actors
0,20124079,"광해, 왕이 된 남자",20120913,"[사극, 드라마]",[추창민],"[이병헌, 류승룡, 한효주, 장광, 김인권]"
1,20183782,기생충,20190530,[드라마],[봉준호],"[송강호, 이선균, 조여정, 최우식, 박소담]"
2,20182530,극한직업,20190123,[코미디],[이병헌],"[류승룡, 이하늬, 진선규, 이동휘, 공명]"


### 테스트 : 해당 날짜의 영화 코드 추출해서 상세 정보 보기

In [19]:
import pandas as pd

df = pd.read_csv("kobis_daily_boxoffice_final.csv", dtype={'movieCd': str, 'date': str})

test_date = "20031111"
test_df = df[df['date'] == test_date]

# 결과 확인
print(f"--- {test_date} 날짜 조회 결과 ---")
print("조회된 행 개수:", len(test_df))

if len(test_df) > 0:
    print("\n데이터 샘플:")
    display(test_df.head())
    
    # movieCd 리스트 뽑기 
    movie_list = test_df['movieCd'].tolist()
    print("\n추출된 movieCd 리스트:", movie_list)
else:
    print("\n해당 날짜의 데이터를 찾을 수 없습니다.")
    print("파일에 실제 들어있는 날짜 샘플:", df['date'].unique()[:5])

--- 20031111 날짜 조회 결과 ---
조회된 행 개수: 7

데이터 샘플:


,rnum,rank,rankInten,rankOldAndNew,movieCd,movieNm,openDt,salesAmt,salesShare,salesInten,salesChange,salesAcc,audiCnt,audiInten,audiChange,audiAcc,scrnCnt,showCnt,date
1,1,1,0,NEW,20030191,매트릭스3 레볼루션,2003-11-05,7676000,62.6,7676000,100.0,7676000,870,870,100.0,870,6,16,20031111
2,2,2,0,NEW,20030152,위대한 유산,2003-10-24,1658000,13.5,1658000,100.0,1658000,177,177,100.0,177,1,3,20031111
3,3,3,0,NEW,20030154,황산벌,2003-10-17,1257000,10.3,1257000,100.0,1257000,116,116,100.0,116,1,3,20031111
4,4,4,0,NEW,20030247,아이덴티티,2003-10-31,821000,6.7,821000,100.0,821000,102,102,100.0,102,1,4,20031111
5,5,5,0,NEW,20030127,스캔들-조선남녀상열지사,2003-10-02,571000,4.7,571000,100.0,571000,60,60,100.0,60,1,3,20031111



추출된 movieCd 리스트: ['20030191', '20030152', '20030154', '20030247', '20030127', '20030284', '20030393']


In [21]:
import requests
import time
import pandas as pd

API_KEY = "e60a0576c6cdfd42b0776f86f25ac236"
url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"

results = []

for movie_cd in movie_list:
    params = {"key": API_KEY, "movieCd": movie_cd}
    
    res = requests.get(url, params=params)
    data = res.json()
    
    if "movieInfoResult" not in data:
        print("중단됨:", data)
        break
    
    info = data["movieInfoResult"]["movieInfo"]
    
    results.append(info)
    
    print("완료:", movie_cd)
    
    time.sleep(0.3)

df_movies = pd.DataFrame(results)
print(df_movies)

완료: 20030191
완료: 20030152
완료: 20030154
완료: 20030247
완료: 20030127
완료: 20030284
완료: 20030393
    movieCd       movieNm                          movieNmEn  \
0  20030191    매트릭스3 레볼루션             The Matrix Revolutions   
1  20030152        위대한 유산           The Greatest Expectation   
2  20030154           황산벌  Once Upon A Time In A Battlefield   
3  20030247         아이덴티티                           Identity   
4  20030127  스캔들-조선남녀상열지사                     Untold Scandal   
5  20030284            깝스                               Kops   
6  20030393    케이트 앤 레오폴드                     Kate & Leopold   

                movieNmOg showTm prdtYear    openDt prdtStatNm typeNm  \
0  The Matrix Revolutions    128     2003  20031105         개봉     장편   
1                            115     2003  20031024         개봉     장편   
2                            104     2003  20031017         개봉     장편   
3                Identity     90     2003  20031031         개봉     장편   
4                            12

데이터 확인 -> 잘 나옴!

In [22]:
df_movies.to_csv("kobis_movie_detail_raw.csv", index=False, encoding="utf-8-sig")

# 2003~2006 데이터 수집

수집 영화 수 : 602개

In [ ]:
import pandas as pd
import requests
import time
import os

API_KEY = "e60a0576c6cdfd42b0776f86f25ac236"
url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"
CHECKPOINT_FILE = "kobis_movie_detail_2003_2006_checkpoint.csv"
FINAL_FILE = "kobis_movie_detail_2003_2006_raw.csv"

# 1. CSV 불러오기
df = pd.read_csv("kobis_daily_boxoffice_final.csv", dtype={'movieCd': str, 'date': str})

# 2. 2003~2006 필터링
df_filtered = df[(df['date'] >= "20030101") & (df['date'] <= "20061231")]

# 3. movieCd 추출 (중복 제거)
movie_list = df_filtered['movieCd'].drop_duplicates().tolist()
total_count = len(movie_list)

print(f"수집 대상 영화 수: {total_count}개")

# 4. 상세정보 수집 시작
results = []

for i, movie_cd in enumerate(movie_list):
    try:
        params = {"key": API_KEY, "movieCd": movie_cd}
        res = requests.get(url, params=params, timeout=10)
        
        if res.status_code != 200:
            print(f"중단됨 (상태코드): {res.status_code}")
            break
        
        data = res.json()
        
        # API 제한(faultInfo) 또는 응답 이상 체크
        if "movieInfoResult" not in data:
            print(f"중단됨 (API 제한 또는 에러): {data.get('faultInfo', data)}")
            break
        
        info = data["movieInfoResult"]["movieInfo"]
        results.append(info)
        
        # 진행 상황 출력
        if (i + 1) % 10 == 0 or (i + 1) == total_count:
            print(f"진행 중: {i+1}/{total_count} 완료 (ID: {movie_cd})")
        
        # 30개마다 중간 저장 (Checkpoint)
        if (i + 1) % 30 == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False, encoding="utf-8-sig")
            print(f"--- {i+1}개 지점 중간 저장 완료! (파일: {CHECKPOINT_FILE}) ---")
        
        time.sleep(0.3) # 서버 과부하 방지
    
    except Exception as e:
        print(f"에러 발생: {e}")
        break

# 5. 수집 완료 후 최종 저장
if results:
    df_movies = pd.DataFrame(results)
    df_movies.to_csv(FINAL_FILE, index=False, encoding="utf-8-sig")
    print("="*50)
    print(f"모든 수집 완료 최종 파일명: {FINAL_FILE}")
    print(f"총 수집 데이터: {len(df_movies)}행")
    
    # 중간 저장 파일은 이제 필요 없으니 삭제
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
else:
    print("수집된 데이터가 없습니다.")

수집 대상 영화 수: 602개
진행 중: 10/602 완료 (ID: 20030395)
진행 중: 20/602 완료 (ID: 20030398)
진행 중: 30/602 완료 (ID: 20030408)
--- 30개 지점 중간 저장 완료! (파일: kobis_movie_detail_2003_2006_checkpoint.csv) ---
진행 중: 40/602 완료 (ID: 20030435)
진행 중: 50/602 완료 (ID: 20030206)
진행 중: 60/602 완료 (ID: 20040448)
--- 60개 지점 중간 저장 완료! (파일: kobis_movie_detail_2003_2006_checkpoint.csv) ---
진행 중: 70/602 완료 (ID: 20040509)
진행 중: 80/602 완료 (ID: 20040528)
진행 중: 90/602 완료 (ID: 20040521)
--- 90개 지점 중간 저장 완료! (파일: kobis_movie_detail_2003_2006_checkpoint.csv) ---
진행 중: 100/602 완료 (ID: 20040555)
진행 중: 110/602 완료 (ID: 20040591)
진행 중: 120/602 완료 (ID: 20040515)
--- 120개 지점 중간 저장 완료! (파일: kobis_movie_detail_2003_2006_checkpoint.csv) ---
진행 중: 130/602 완료 (ID: 20040586)
진행 중: 140/602 완료 (ID: 20040618)
진행 중: 150/602 완료 (ID: 20040554)
--- 150개 지점 중간 저장 완료! (파일: kobis_movie_detail_2003_2006_checkpoint.csv) ---
진행 중: 160/602 완료 (ID: 20040619)
진행 중: 170/602 완료 (ID: 20040686)
진행 중: 180/602 완료 (ID: 20040695)
--- 180개 지점 중간 저장 완료! (파일: kobis_movie_

# 2007~2012 데이터 수집

수집 영화 수 : 1228개

In [ ]:
import pandas as pd
import requests
import time
import os

API_KEY = "e60a0576c6cdfd42b0776f86f25ac236"
url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"
CHECKPOINT_FILE = "kobis_movie_detail_2007_2012_checkpoint.csv"
FINAL_FILE = "kobis_movie_detail_2007_2012_raw.csv"

# 1. CSV 불러오기
df = pd.read_csv("kobis_daily_boxoffice_final.csv", dtype={'movieCd': str, 'date': str})

# 2. 2007~2012 필터링
df_filtered = df[(df['date'] >= "20070101") & (df['date'] <= "20121231")]

# 3. movieCd 추출 (중복 제거)
movie_list = df_filtered['movieCd'].drop_duplicates().tolist()
total_count = len(movie_list)

print(f"수집 대상 영화 수: {total_count}개")

# 4. 상세정보 수집 시작
results = []

for i, movie_cd in enumerate(movie_list):
    try:
        params = {"key": API_KEY, "movieCd": movie_cd}
        res = requests.get(url, params=params, timeout=10)
        
        if res.status_code != 200:
            print(f"중단됨 (상태코드): {res.status_code}")
            break
        
        data = res.json()
        
        # API 제한(faultInfo) 또는 응답 이상 체크
        if "movieInfoResult" not in data:
            print(f"중단됨 (API 제한 또는 에러): {data.get('faultInfo', data)}")
            break
        
        info = data["movieInfoResult"]["movieInfo"]
        results.append(info)
        
        # 진행 상황 출력
        if (i + 1) % 10 == 0 or (i + 1) == total_count:
            print(f"진행 중: {i+1}/{total_count} 완료 (ID: {movie_cd})")
        
        # 30개마다 중간 저장 (Checkpoint)
        if (i + 1) % 30 == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False, encoding="utf-8-sig")
            print(f"--- {i+1}개 지점 중간 저장 완료! (파일: {CHECKPOINT_FILE}) ---")
        
        time.sleep(0.4) # 서버 과부하 방지
    
    except Exception as e:
        print(f"에러 발생: {e}")
        break

# 5. 수집 완료 후 최종 저장
if results:
    df_movies = pd.DataFrame(results)
    df_movies.to_csv(FINAL_FILE, index=False, encoding="utf-8-sig")
    print("="*50)
    print(f"모든 수집 완료 최종 파일명: {FINAL_FILE}")
    print(f"총 수집 데이터: {len(df_movies)}행")
    
    # 중간 저장 파일은 이제 필요 없으니 삭제
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
else:
    print("수집된 데이터가 없습니다.")

수집 대상 영화 수: 1228개
진행 중: 10/1228 완료 (ID: 20060392)
진행 중: 20/1228 완료 (ID: 20060391)
진행 중: 30/1228 완료 (ID: 20060397)
--- 30개 지점 중간 저장 완료! (파일: kobis_movie_detail_2007_2012_checkpoint.csv) ---
진행 중: 40/1228 완료 (ID: 20060422)
진행 중: 50/1228 완료 (ID: 20070036)
진행 중: 60/1228 완료 (ID: 20070102)
--- 60개 지점 중간 저장 완료! (파일: kobis_movie_detail_2007_2012_checkpoint.csv) ---
진행 중: 70/1228 완료 (ID: 20070111)
진행 중: 80/1228 완료 (ID: 20070056)
진행 중: 90/1228 완료 (ID: 20060230)
--- 90개 지점 중간 저장 완료! (파일: kobis_movie_detail_2007_2012_checkpoint.csv) ---
진행 중: 100/1228 완료 (ID: 20070196)
진행 중: 110/1228 완료 (ID: 20060263)
진행 중: 120/1228 완료 (ID: 20070254)
--- 120개 지점 중간 저장 완료! (파일: kobis_movie_detail_2007_2012_checkpoint.csv) ---
진행 중: 130/1228 완료 (ID: 20060425)
진행 중: 140/1228 완료 (ID: 20070064)
진행 중: 150/1228 완료 (ID: 20070421)
--- 150개 지점 중간 저장 완료! (파일: kobis_movie_detail_2007_2012_checkpoint.csv) ---
진행 중: 160/1228 완료 (ID: 20070023)
진행 중: 170/1228 완료 (ID: 20070497)
진행 중: 180/1228 완료 (ID: 20070541)
--- 180개 지점 중간 저장 완료

In [26]:
print(total_count)

1228


# 2013 ~ 2019 데이터 수집

수집 영화 수 : 1783개

In [27]:
import pandas as pd
import requests
import time
import os

API_KEY = "e60a0576c6cdfd42b0776f86f25ac236"
url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"
CHECKPOINT_FILE = "kobis_movie_detail_2013_2019_checkpoint.csv"
FINAL_FILE = "kobis_movie_detail_2013_2019_raw.csv"

# 1. CSV 불러오기
df = pd.read_csv("kobis_daily_boxoffice_final.csv", dtype={'movieCd': str, 'date': str})

# 2. 2013~2019 필터링
df_filtered = df[(df['date'] >= "20130101") & (df['date'] <= "20191231")]

# 3. movieCd 추출 (중복 제거)
movie_list = df_filtered['movieCd'].drop_duplicates().tolist()
total_count = len(movie_list)

print(f"수집 대상 영화 수: {total_count}개")

# 4. 상세정보 수집 시작
results = []

for i, movie_cd in enumerate(movie_list):
    try:
        params = {"key": API_KEY, "movieCd": movie_cd}
        res = requests.get(url, params=params, timeout=10)
        
        if res.status_code != 200:
            print(f"중단됨 (상태코드): {res.status_code}")
            break
        
        data = res.json()
        
        # API 제한(faultInfo) 또는 응답 이상 체크
        if "movieInfoResult" not in data:
            print(f"중단됨 (API 제한 또는 에러): {data.get('faultInfo', data)}")
            break
        
        info = data["movieInfoResult"]["movieInfo"]
        results.append(info)
        
        # 진행 상황 출력
        if (i + 1) % 10 == 0 or (i + 1) == total_count:
            print(f"진행 중: {i+1}/{total_count} 완료 (ID: {movie_cd})")
        
        # 30개마다 중간 저장 (Checkpoint)
        if (i + 1) % 30 == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False, encoding="utf-8-sig")
            print(f"--- {i+1}개 지점 중간 저장 완료! (파일: {CHECKPOINT_FILE}) ---")
        
        time.sleep(0.4) # 서버 과부하 방지
    
    except Exception as e:
        print(f"에러 발생: {e}")
        break

# 5. 수집 완료 후 최종 저장
if results:
    df_movies = pd.DataFrame(results)
    df_movies.to_csv(FINAL_FILE, index=False, encoding="utf-8-sig")
    print("="*50)
    print(f"모든 수집 완료 최종 파일명: {FINAL_FILE}")
    print(f"총 수집 데이터: {len(df_movies)}행")
    
    # 중간 저장 파일은 이제 필요 없으니 삭제 (깔끔하게 정리)
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
else:
    print("수집된 데이터가 없습니다.")

수집 대상 영화 수: 1783개
진행 중: 10/1783 완료 (ID: 20124936)
진행 중: 20/1783 완료 (ID: 20123681)
진행 중: 30/1783 완료 (ID: 20010292)
--- 30개 지점 중간 저장 완료! (파일: kobis_movie_detail_2013_2019_checkpoint.csv) ---
진행 중: 40/1783 완료 (ID: 20122843)
진행 중: 50/1783 완료 (ID: 20128690)
진행 중: 60/1783 완료 (ID: 20126671)
--- 60개 지점 중간 저장 완료! (파일: kobis_movie_detail_2013_2019_checkpoint.csv) ---
진행 중: 70/1783 완료 (ID: 20120381)
진행 중: 80/1783 완료 (ID: 20136741)
진행 중: 90/1783 완료 (ID: 20135221)
--- 90개 지점 중간 저장 완료! (파일: kobis_movie_detail_2013_2019_checkpoint.csv) ---
진행 중: 100/1783 완료 (ID: 20137181)
진행 중: 110/1783 완료 (ID: 20120103)
진행 중: 120/1783 완료 (ID: 20134044)
--- 120개 지점 중간 저장 완료! (파일: kobis_movie_detail_2013_2019_checkpoint.csv) ---
진행 중: 130/1783 완료 (ID: 20137549)
진행 중: 140/1783 완료 (ID: 20136049)
진행 중: 150/1783 완료 (ID: 20139265)
--- 150개 지점 중간 저장 완료! (파일: kobis_movie_detail_2013_2019_checkpoint.csv) ---
진행 중: 160/1783 완료 (ID: 20137562)
진행 중: 170/1783 완료 (ID: 20130783)
진행 중: 180/1783 완료 (ID: 20122785)
--- 180개 지점 중간 저장 완료

api 호출 제한으로 인해 마저 다시 이어서 진행

In [28]:
import pandas as pd
import requests
import time
import os

API_KEY = "c39ef9e7a0d75f40f96a25e6672ddf8c"
url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"

CHECKPOINT_FILE = "kobis_movie_detail_2013_2019_checkpoint_2.csv"
FINAL_FILE = "kobis_movie_detail_2013_2019_raw_2.csv"

# 1. 전체 movie_list 다시 만들기
df = pd.read_csv("kobis_daily_boxoffice_final.csv", dtype={'movieCd': str, 'date': str})
df_filtered = df[(df['date'] >= "20130101") & (df['date'] <= "20191231")]
movie_list = df_filtered['movieCd'].drop_duplicates().tolist()

# 2. 이미 수집한 데이터 불러오기
df_done = pd.read_csv("kobis_movie_detail_2013_2019_raw.csv", dtype={'movieCd': str})
done_list = df_done['movieCd'].tolist()

# 3. 남은 영화만 필터링
remaining_list = [m for m in movie_list if m not in done_list]

print(f"남은 영화 수: {len(remaining_list)}개")

# 4. 이어서 수집
results = []

for i, movie_cd in enumerate(remaining_list):
    try:
        params = {"key": API_KEY, "movieCd": movie_cd}
        res = requests.get(url, params=params, timeout=10)
        
        if res.status_code != 200:
            print(f"중단됨 (상태코드): {res.status_code}")
            break
        
        data = res.json()
        
        if "movieInfoResult" not in data:
            print(f"중단됨 (API 제한): {data.get('faultInfo', data)}")
            break
        
        info = data["movieInfoResult"]["movieInfo"]
        results.append(info)
        
        if (i + 1) % 10 == 0:
            print(f"진행 중: {i+1}/{len(remaining_list)} 완료")
        
        if (i + 1) % 30 == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False, encoding="utf-8-sig")
            print(f"{i+1}개 중간 저장 완료")
        
        time.sleep(0.4)
    
    except Exception as e:
        print("에러 발생:", e)
        break

# 5. 기존 데이터 + 새 데이터 합치기
if results:
    df_new = pd.DataFrame(results)
    df_final = pd.concat([df_done, df_new], ignore_index=True)
    
    df_final.to_csv("kobis_movie_detail_2013_2019_final.csv", index=False, encoding="utf-8-sig")
    
    print("="*50)
    print("전체 데이터 완성!")
    print("총 행 수:", len(df_final))
else:
    print("추가 수집된 데이터 없음")

남은 영화 수: 624개
진행 중: 10/624 완료
진행 중: 20/624 완료
진행 중: 30/624 완료
30개 중간 저장 완료
진행 중: 40/624 완료
진행 중: 50/624 완료
진행 중: 60/624 완료
60개 중간 저장 완료
진행 중: 70/624 완료
진행 중: 80/624 완료
진행 중: 90/624 완료
90개 중간 저장 완료
진행 중: 100/624 완료
진행 중: 110/624 완료
진행 중: 120/624 완료
120개 중간 저장 완료
진행 중: 130/624 완료
진행 중: 140/624 완료
진행 중: 150/624 완료
150개 중간 저장 완료
진행 중: 160/624 완료
진행 중: 170/624 완료
진행 중: 180/624 완료
180개 중간 저장 완료
진행 중: 190/624 완료
진행 중: 200/624 완료
진행 중: 210/624 완료
210개 중간 저장 완료
진행 중: 220/624 완료
진행 중: 230/624 완료
진행 중: 240/624 완료
240개 중간 저장 완료
진행 중: 250/624 완료
진행 중: 260/624 완료
진행 중: 270/624 완료
270개 중간 저장 완료
진행 중: 280/624 완료
진행 중: 290/624 완료
진행 중: 300/624 완료
300개 중간 저장 완료
진행 중: 310/624 완료
진행 중: 320/624 완료
진행 중: 330/624 완료
330개 중간 저장 완료
진행 중: 340/624 완료
진행 중: 350/624 완료
진행 중: 360/624 완료
360개 중간 저장 완료
진행 중: 370/624 완료
진행 중: 380/624 완료
진행 중: 390/624 완료
390개 중간 저장 완료
진행 중: 400/624 완료
진행 중: 410/624 완료
진행 중: 420/624 완료
420개 중간 저장 완료
진행 중: 430/624 완료
진행 중: 440/624 완료
진행 중: 450/624 완료
450개 중간 저장 완료
진행 중: 460/624 완료
진행 중: 

In [30]:
print(total_count)

1783


# 2020 ~ 2025 데이터 수집

수집 영화 수 : 1641개

In [ ]:
import pandas as pd
import requests
import time
import os

API_KEY = "c39ef9e7a0d75f40f96a25e6672ddf8c"
url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"
CHECKPOINT_FILE = "kobis_movie_detail_2020_2025_checkpoint.csv"
FINAL_FILE = "kobis_movie_detail_2020_2025_raw.csv"

# 1. CSV 불러오기
df = pd.read_csv("kobis_daily_boxoffice_final.csv", dtype={'movieCd': str, 'date': str})

# 2. 2020~2025 필터링
df_filtered = df[(df['date'] >= "20200101") & (df['date'] <= "20251231")]

# 3. movieCd 추출 (중복 제거)
movie_list = df_filtered['movieCd'].drop_duplicates().tolist()
total_count = len(movie_list)

print(f"수집 대상 영화 수: {total_count}개")

# 4. 상세정보 수집 시작
results = []

for i, movie_cd in enumerate(movie_list):
    try:
        params = {"key": API_KEY, "movieCd": movie_cd}
        res = requests.get(url, params=params, timeout=10)
        
        if res.status_code != 200:
            print(f"중단됨 (상태코드): {res.status_code}")
            break
        
        data = res.json()
        
        # API 제한(faultInfo) 또는 응답 이상 체크
        if "movieInfoResult" not in data:
            print(f"중단됨 (API 제한 또는 에러): {data.get('faultInfo', data)}")
            break
        
        info = data["movieInfoResult"]["movieInfo"]
        results.append(info)
        
        # 진행 상황 출력
        if (i + 1) % 10 == 0 or (i + 1) == total_count:
            print(f"진행 중: {i+1}/{total_count} 완료 (ID: {movie_cd})")
        
        # 30개마다 중간 저장 (Checkpoint)
        if (i + 1) % 30 == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False, encoding="utf-8-sig")
            print(f"--- {i+1}개 지점 중간 저장 완료! (파일: {CHECKPOINT_FILE}) ---")
        
        time.sleep(0.4) # 서버 과부하 방지
    
    except Exception as e:
        print(f"에러 발생: {e}")
        break

# 5. 수집 완료 후 최종 저장
if results:
    df_movies = pd.DataFrame(results)
    df_movies.to_csv(FINAL_FILE, index=False, encoding="utf-8-sig")
    print("="*50)
    print(f"모든 수집 완료 최종 파일명: {FINAL_FILE}")
    print(f"총 수집 데이터: {len(df_movies)}행")
    
    # 중간 저장 파일은 이제 필요 없으니 삭제 
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
else:
    print("수집된 데이터가 없습니다.")

수집 대상 영화 수: 1641개
진행 중: 10/1641 완료 (ID: 20199981)
진행 중: 20/1641 완료 (ID: 20198182)
진행 중: 30/1641 완료 (ID: 20180962)
--- 30개 지점 중간 저장 완료! (파일: kobis_movie_detail_2020_2025_checkpoint.csv) ---
진행 중: 40/1641 완료 (ID: 20192399)
진행 중: 50/1641 완료 (ID: 20204681)
진행 중: 60/1641 완료 (ID: 20192751)
--- 60개 지점 중간 저장 완료! (파일: kobis_movie_detail_2020_2025_checkpoint.csv) ---
진행 중: 70/1641 완료 (ID: 20206141)
진행 중: 80/1641 완료 (ID: 20128588)
진행 중: 90/1641 완료 (ID: 20206165)
--- 90개 지점 중간 저장 완료! (파일: kobis_movie_detail_2020_2025_checkpoint.csv) ---
진행 중: 100/1641 완료 (ID: 20181685)
진행 중: 110/1641 완료 (ID: 20204501)
진행 중: 120/1641 완료 (ID: 20208265)
--- 120개 지점 중간 저장 완료! (파일: kobis_movie_detail_2020_2025_checkpoint.csv) ---
진행 중: 130/1641 완료 (ID: 20183813)
진행 중: 140/1641 완료 (ID: 20188124)
진행 중: 150/1641 완료 (ID: 20204644)
--- 150개 지점 중간 저장 완료! (파일: kobis_movie_detail_2020_2025_checkpoint.csv) ---
진행 중: 160/1641 완료 (ID: 20112703)
진행 중: 170/1641 완료 (ID: 20209961)
진행 중: 180/1641 완료 (ID: 20190735)
--- 180개 지점 중간 저장 완료

In [32]:
print(total_count)

1641


# 수집한 데이터(2003~2025) 합치기

In [99]:
import pandas as pd

files = [
    "kobis_movie_detail_2003_2006_raw.csv",
    "kobis_movie_detail_2007_2012_raw.csv",
    "kobis_movie_detail_2013_2019_final.csv", #이 데이터는 중간에 api 키가 다 닳아서 따로 합쳐줬기 때문에 이름이 final
    "kobis_movie_detail_2020_2025_raw.csv"
]

df_all = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print("전체 행 개수:", len(df_all))
print("고유 영화 수:", df_all['movieCd'].nunique())

전체 행 개수: 5254
고유 영화 수: 5220


#### 기간을 나눠 데이터 수집했기 때문에 중복 집계 되었을 가능성

중복제거

In [100]:
df_all = df_all.drop_duplicates(subset='movieCd')

In [101]:
print("행 개수:", df_all.shape[0])
print("고유 movieCd:", df_all['movieCd'].nunique())

행 개수: 5221
고유 movieCd: 5220


In [102]:
# movieCd(영화코드)가 비어있는 행 1개 찾기
ghost_row = df_all[df_all['movieCd'].isna()]
print(ghost_row)

    movieCd movieNm movieNmEn movieNmOg  showTm  prdtYear  openDt prdtStatNm  \
623     NaN     NaN       NaN       NaN     NaN       NaN     NaN        NaN   

    typeNm nations genres directors actors showTypes companys audits staffs  
623    NaN      []     []        []     []        []       []     []     []  


유령 행 삭제

In [103]:
df_all = df_all.dropna(subset=['movieCd'])
print("최종 행 개수:", df_all.shape[0])

최종 행 개수: 5220


타입이 달라서 다르게 집계된 행 정리

In [ ]:
# 영화 코드 타입 맞춰줬을 때 중복 확인
temp_cd_str = df_all['movieCd'].astype(str).str.strip()
duplicates_mask = temp_cd_str.duplicated(keep=False) # keep=False는 원본과 복사본 모두 가져오라는 뜻

# 중복된 데이터들만 따로 모읍니다.
df_duplicates = df_all[duplicates_mask].copy()

# 타입확인
df_duplicates['원래_데이터_타입'] = df_duplicates['movieCd'].apply(lambda x: type(x).__name__)

#정렬
df_duplicates['정렬용_코드'] = temp_cd_str[duplicates_mask]
df_duplicates = df_duplicates.sort_values(by='정렬용_코드')

# 5쌍만 보기
print(f"중복 발견! 쌍으로 존재하는 데이터 : {len(df_duplicates)}개")
print("쌍으로 존재하는 데이터 보기")
print(df_duplicates[['movieCd', '원래_데이터_타입', 'movieNm']].head(10))

중복 발견! 쌍으로 존재하는 데이터 : 216개
쌍으로 존재하는 데이터 보기
       movieCd 원래_데이터_타입    movieNm
2707  19900204       int  죽은 시인의 사회
4028  19900204       str  죽은 시인의 사회
1897  19950095       int         레옹
4801  19950095       str         레옹
2621  19960126       int    비포 선라이즈
4866  19960126       str    비포 선라이즈
1677  19980074       str       타이타닉
3104  19980074       int       타이타닉
2622  19990140       int   인생은 아름다워
5110  19990140       str   인생은 아름다워


In [109]:
df_all['movieCd'] = df_all['movieCd'].astype(str).str.strip()
df_all = df_all.drop_duplicates(subset=['movieCd'])

print("최종 데이터 개수:", df_all.shape[0])
print("고유한 영화 코드 개수:", df_all['movieCd'].nunique())

df_all.to_csv("kobis_movie_detail_full.csv", index=False, encoding="utf-8-sig")
print("파일 정리 완료!")

최종 데이터 개수: 5112
고유한 영화 코드 개수: 5112
파일 정리 완료!
